In [1]:
from common import MODEL, banner, get_client

banner("TEST")

client = get_client()


======== TEST ========


In [2]:
client

In [27]:
banner("Step#1 Single Cell")
r1 = client.messages.create(model=MODEL,
                              max_tokens=300,
                              system = "You are an Concise assistant for enterprise IT consultant",
                              messages= [{"role":"user","content":"In a 2 sentence:What is prompt caching?"}]  )
print(r1.content[0].text)





======== Step#1 Single Cell ========
Prompt caching is a feature in LLM APIs that stores the processed state of a repeated prompt prefix (such as system instructions, documents, or tool definitions) so subsequent requests can reuse it instead of reprocessing it from scratch. This reduces latency and cost, often significantly, for workloads with large, stable context and only small changes per request.


In [30]:
banner("Step#2 Single Cell")
resp = client.messages.create(model=MODEL,
                              max_tokens=300,
                              system = "You are an Concise assistant for enterprise IT consultant",
                              messages= [{"role":"user","content":"In a 2 sentence:What is prompt caching?"},{"role":"assistant","content":r1.content[0].text},{"role":"user","content":"What is API"}]  )
print(resp.content[0].text)


======== Step#2 Single Cell ========
An API (Application Programming Interface) is a defined set of rules and endpoints that lets one software system request data or services from another without needing to know its internal workings. In enterprise IT, APIs are the standard way to integrate applications, expose services, and automate workflows across systems.


In [ ]:
import json

banner("Step#3 Structured Output")

raw = client.messages.create(model=MODEL,
                             max_tokens=400,
                             system = "Reply with only JSON object, no prose, no code fences",
                             messages = [{"role":"user","content":"Extract: 'Order 4471 from Acme, 3 laptops, urgent.'"
                                                                   "Key: customer(str), quantity(int), urgent(bool). "}])
text = raw.content[0].text.strip()

try:
    data = json.loads(text)
    missing = {"order_id","customer","quantity","urgent"} - data.keys()
    assert not missing, f"missing keys: {missing}"
    print(f"parsed ok:{data}")
except (json.JSONDecodeError,AssertionError) as exc:
    print("Validation failed", exc, "/nraw",text)
if raw.stop_reason == "max_tokens":
    print("WARNING: output truncated; do not trust this json") 


======== Step#3 Structured Output ========
Validation failed missing keys: {'order_id'} /nraw {"customer": "Acme", "quantity": 3, "urgent": true}
